# NB17 — Coupling bootstrap CIs + EfficientNet dispersion (full archive)

Two jobs, both feeding the CDTS manuscript:

**Job A — per-architecture coupling CIs.** Bootstrap 95% CIs for the competence–calibration coupling on each detector (Xception −0.88, EfficientNet −0.83, CLIP −0.859). These back the rigor claim that *the central coupling carries bootstrap CIs on all three architectures*, not just the pooled number.

**Job B — EfficientNet dispersion over the full DF40 set.** Table 9 currently rests on 8 retained generators and calls the cell *near-constant / flat*. This recomputes dispersion-vs-competence over all 20 generators: either it confirms flat (and the "8 retained" caveat can be dropped) or it returns the true value.

**Design note.** Job A bootstraps the **committed per-generator (AUC, ECE) points** already saved in `reports/calibration/` — the same points that produced the headline r values — so the CIs are on exactly the numbers in the paper. It asserts each point estimate matches the headline before reporting its CI. It does **not** re-score from parquets. Job B re-scores **only** EfficientNet, because its full per-generator dispersion was never saved.

Run order: Cell 0 (setup) → Cell 1 (Job A) → Cell 2 (Job B) → Cell 3 (commit). Paste the Cell 1 table and Cell 2 verdict back into the chat.

In [1]:
# CELL 0 — setup: restore git identity + mount Drive + config
import os, shutil, subprocess
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

CDTS_ROOT = '/content/drive/MyDrive/CDTS_Research'
REPO      = f'{CDTS_ROOT}/deepfake-trust-research'

# restore git creds from Drive parent into /root/ (prevents "Author identity unknown")
for src, dst in [(f'{CDTS_ROOT}/.git-credentials', '/root/.git-credentials'),
                 (f'{CDTS_ROOT}/.gitconfig',       '/root/.gitconfig')]:
    if os.path.exists(src):
        shutil.copy(src, dst)
print('git identity restored' if os.path.exists('/root/.gitconfig') else 'WARNING: no .gitconfig in Drive')

# config: every path from here, never hardcoded downstream
CFG = dict(
    repo    = REPO,
    reports = f'{REPO}/reports/calibration',
    scores  = f'{REPO}/reports/scores',
    n_boot  = 5000,
    n_bins  = 15,      # equal-mass ECE
    seed    = 42,
)
os.chdir(CFG['repo'])
print('cwd:', os.getcwd())

Mounted at /content/drive
git identity restored
cwd: /content/drive/MyDrive/CDTS_Research/deepfake-trust-research


## Job A — per-architecture coupling bootstrap CIs

Bootstraps the committed per-generator points. Each architecture's point estimate is asserted against the manuscript headline **before** the CI is trusted: if an assertion fails, that CSV is not the file behind the headline.

In [2]:
# CELL 1 — JOB A: bootstrap CIs for the per-architecture coupling
import pandas as pd, numpy as np
from scipy.stats import pearsonr

def boot_ci_r(x, y, n_boot=CFG['n_boot'], seed=CFG['seed']):
    """Percentile bootstrap 95% CI for Pearson r over paired points."""
    x = np.asarray(x, float); y = np.asarray(y, float)
    rng = np.random.RandomState(seed); rs = []; n = len(x)
    for _ in range(n_boot):
        idx = rng.randint(0, n, n)
        if np.std(x[idx]) > 0 and np.std(y[idx]) > 0:
            rs.append(pearsonr(x[idx], y[idx])[0])
    lo, hi = np.percentile(rs, [2.5, 97.5])
    return float(np.mean(rs)), float(lo), float(hi)

# Each architecture's per-generator coupling points live in a committed CSV.
ARCH_SOURCES = {
    # label             file                               headline r   tol
    'Xception':        ('unified_trust_signals.csv',      -0.88,       0.02),
    'CLIP-ViT':        ('unified_trust_signals_clip.csv', -0.859,      0.02),
    # EfficientNet headline is -0.83; the committed DF40 file is 8 gens (-0.71).
    'EfficientNet-B4': ('coupling_effnetb4_df40.csv',     None,        None),
}

rows = []
for arch, (fname, headline, tol) in ARCH_SOURCES.items():
    d = pd.read_csv(f"{CFG['reports']}/{fname}")
    a, e = d['AUC'].to_numpy(float), d['ECE_cal'].to_numpy(float)
    r = pearsonr(a, e)[0]
    if headline is not None:
        assert abs(r - headline) < tol, (
            f"{arch}: committed r={r:.3f} != headline {headline} "
            f"-> {fname} is not the file behind the headline; do NOT report this CI")
    rmean, lo, hi = boot_ci_r(a, e)
    rows.append(dict(architecture=arch, n=len(d), r=round(r, 3),
                     ci_lo=round(lo, 2), ci_hi=round(hi, 2), source=fname))
    print(f"{arch:16s}  r = {r:+.3f}   95% CI [{lo:+.2f}, {hi:+.2f}]   n={len(d):2d}  ({fname})")

# pooled (32 configs) — already in the manuscript, recomputed for the record
g = pd.read_csv(f"{CFG['reports']}/coupling_full_grid.csv")
rp = pearsonr(g['AUC'], g['ECE_cal'])[0]
rpm, plo, phi = boot_ci_r(g['AUC'], g['ECE_cal'])
print(f"\n{'POOLED (32 cfg)':16s}  r = {rp:+.3f}   95% CI [{plo:+.2f}, {phi:+.2f}]   n={len(g)}")
rows.append(dict(architecture='POOLED', n=len(g), r=round(rp, 3),
                 ci_lo=round(plo, 2), ci_hi=round(phi, 2), source='coupling_full_grid.csv'))

ci_df = pd.DataFrame(rows)
out_a = f"{CFG['reports']}/coupling_bootstrap_CIs.csv"
ci_df.to_csv(out_a, index=False)
print(f"\nsaved -> {out_a}")
print("\nNOTE: if EfficientNet n < 20, its CI is on the retained subset; run Cell 2")
print("to score the full archive and recompute the EffNet coupling on 20 gens.")
ci_df

Xception          r = -0.880   95% CI [-0.95, -0.79]   n=20  (unified_trust_signals.csv)
CLIP-ViT          r = -0.859   95% CI [-0.93, -0.75]   n=20  (unified_trust_signals_clip.csv)
EfficientNet-B4   r = -0.707   95% CI [-0.96, -0.29]   n= 8  (coupling_effnetb4_df40.csv)

POOLED (32 cfg)   r = -0.807   95% CI [-0.90, -0.70]   n=32

saved -> /content/drive/MyDrive/CDTS_Research/deepfake-trust-research/reports/calibration/coupling_bootstrap_CIs.csv

NOTE: if EfficientNet n < 20, its CI is on the retained subset; run Cell 2
to score the full archive and recompute the EffNet coupling on 20 gens.


,architecture,n,r,ci_lo,ci_hi,source
0,Xception,20,-0.880,-0.95,-0.79,unified_trust_signals.csv
1,CLIP-ViT,20,-0.859,-0.93,-0.75,unified_trust_signals_clip.csv
2,EfficientNet-B4,8,-0.707,-0.96,-0.29,coupling_effnetb4_df40.csv
3,POOLED,32,-0.807,-0.90,-0.70,coupling_full_grid.csv


## Job B — EfficientNet dispersion over the full DF40 archive

Re-scores EfficientNet per generator (its full per-generator dispersion was never saved) using the same protocol as the rest of the paper: equal-mass 15-bin ECE, identity-disjoint 50/50 split, hybrid Platt/isotonic calibrator.

If the glob finds fewer than ~12 EffNet parquets, the full archive is not on this Drive — re-run the EffNet DF40 scoring pass first (same `inference.py`: RGB, resize 256 BILINEAR, normalize mean=std=0.5), then re-run this cell.

In [3]:
# CELL 2 — JOB B: EfficientNet score dispersion over the FULL DF40 archive
import glob
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.isotonic import IsotonicRegression

def ece_equalmass(p, y, n_bins=CFG['n_bins']):
    p = np.asarray(p, float); y = np.asarray(y, float)
    o = np.argsort(p); p, y = p[o], y[o]; e = 0.0
    for b in np.array_split(np.arange(len(p)), n_bins):
        if len(b):
            e += len(b) / len(p) * abs(p[b].mean() - y[b].mean())
    return e

def oracle_ece(d, seed=0):
    """50/50 identity-disjoint oracle calibration (hybrid Platt/isotonic), eval-half ECE."""
    rng = np.random.RandomState(seed)
    ids = d['identity_id'].astype(str).unique(); rng.shuffle(ids)
    cal_ids = set(ids[:len(ids) // 2])
    cal = d[d['identity_id'].astype(str).isin(cal_ids)]
    ev  = d[~d['identity_id'].astype(str).isin(cal_ids)]
    if cal['label'].nunique() < 2 or len(ev) < 10:
        return None
    if len(cal) < 1000:
        m = LogisticRegression(C=1e6).fit(cal[['prob_fake']], cal['label'])
        pe = m.predict_proba(ev[['prob_fake']])[:, 1]
    else:
        m = IsotonicRegression(out_of_bounds='clip').fit(cal['prob_fake'], cal['label'])
        pe = m.predict(ev['prob_fake'])
    return ece_equalmass(pe, ev['label'].values)

# Find every EfficientNet DF40 per-generator score parquet.
# Adjust the glob if your EffNet scores use a different prefix.
eff_files = sorted(glob.glob(f"{CFG['scores']}/effnetb4_df40_*.parquet"))
print(f"EfficientNet DF40 score parquets found: {len(eff_files)}")
if len(eff_files) < 12:
    print("  -> fewer than 12: the full archive may not be on this Drive.")
    print("     Re-run the EffNet DF40 scoring pass (same inference.py: RGB,")
    print("     resize 256 BILINEAR, normalize mean=std=0.5), then re-run this cell.")

rows = []
for f in eff_files:
    gen = os.path.basename(f).replace('effnetb4_df40_', '').replace('.parquet', '')
    d = pd.read_parquet(f)
    if d['label'].nunique() < 2:
        continue
    auc = roc_auc_score(d['label'], d['prob_fake'])
    dispersion = float(d['prob_fake'].std())          # same statistic as the Xcep/CLIP cells
    p = d['prob_fake'].clip(1e-6, 1 - 1e-6)
    entropy = float((-(p * np.log(p) + (1 - p) * np.log(1 - p))).mean())
    ece = oracle_ece(d)
    rows.append(dict(method=gen, AUC=round(auc, 4), dispersion=round(dispersion, 4),
                     entropy=round(entropy, 4),
                     ECE_cal=None if ece is None else round(ece, 4), n=len(d)))

E = pd.DataFrame(rows).sort_values('AUC').reset_index(drop=True)
print(f"\nEfficientNet over {len(E)} generators:")
print(E.to_string(index=False))

# the three numbers the manuscript needs
disp_r, disp_lo, disp_hi = boot_ci_r(E['AUC'], E['dispersion'])
ent_r,  _,       _       = boot_ci_r(E['AUC'], E['entropy'])
disp_span = E['dispersion'].max() - E['dispersion'].min()
print(f"\nDISPERSION vs AUC: r = {disp_r:+.3f}  95% CI [{disp_lo:+.2f}, {disp_hi:+.2f}]  n={len(E)}")
print(f"  dispersion range [{E['dispersion'].min():.3f}, {E['dispersion'].max():.3f}]  span={disp_span:.3f}")
print(f"ENTROPY    vs AUC: r = {ent_r:+.3f}   (manuscript Table 9 EffNet entropy = -0.38)")
print()
if disp_span < 0.10:
    print(f"VERDICT: dispersion is NEAR-CONSTANT over {len(E)} generators (span {disp_span:.3f}).")
    print("  -> Table 9 'flat' is confirmed on the full set; the '8 retained' caveat")
    print("     in the caption can be replaced with the full-n near-constant statement.")
else:
    print(f"VERDICT: dispersion VARIES over {len(E)} generators (span {disp_span:.3f}).")
    print(f"  -> replace the Table 9 EffNet dispersion cell with r = {disp_r:.2f} (n={len(E)}),")
    print("     and update Section 7.2 accordingly. Check the sign before claiming consistency.")

# if the full archive is present, also recompute the EffNet COUPLING on 20 gens
valid = E.dropna(subset=['ECE_cal'])
if len(valid) >= 12:
    cr, clo, chi = boot_ci_r(valid['AUC'], valid['ECE_cal'])
    print(f"\nEfficientNet coupling on {len(valid)} gens: r = {cr:+.3f}  95% CI [{clo:+.2f}, {chi:+.2f}]")
    print("  (compare to the headline -0.83; if it matches, use this CI in Cell 1's table)")

out_b = f"{CFG['reports']}/effnet_dispersion_full.csv"
E.to_csv(out_b, index=False)
print(f"\nsaved -> {out_b}")

EfficientNet DF40 score parquets found: 20

EfficientNet over 20 generators:
     method    AUC  dispersion  entropy  ECE_cal     n
  StyleGAN2 0.4850      0.3229   0.3326   0.1925 33794
     pixart 0.4895      0.3632   0.2907   0.1741 33794
  StyleGAN3 0.4947      0.3286   0.3355   0.1897 33794
  sadtalker 0.4959      0.2991   0.3599   0.0560 26606
 StyleGANXL 0.4991      0.2943   0.3703   0.0928 33794
        DiT 0.5305      0.3094   0.3794   0.0361 33794
    wav2lip 0.5350      0.3364   0.3481   0.0816 26434
        SiT 0.5555      0.3157   0.3893   0.0595 33794
        lia 0.6137      0.3350   0.3927   0.0620 25426
   pirender 0.6357      0.3064   0.4435   0.0404 25805
facevid2vid 0.6661      0.3124   0.4473   0.0508 25510
       fomm 0.7180      0.3110   0.4617   0.0717 25898
     inswap 0.7434      0.3581   0.3764   0.0505 19053
       ddim 0.7640      0.3498   0.3791   0.0253 33794
      sd2.1 0.7928      0.3570   0.3387   0.0199 33794
 facedancer 0.8135      0.3672   0.3131   0

## Cell 3 — commit + push (end-of-unit ritual)

Commits only the two new report CSVs and pushes. Then paste the Cell 1 table and the Cell 2 verdict back into the chat for the manuscript edits.

In [4]:
# CELL 3 — commit + push the two new reports
subprocess.run(['git', 'add',
                f"{CFG['reports']}/coupling_bootstrap_CIs.csv",
                f"{CFG['reports']}/effnet_dispersion_full.csv"], cwd=CFG['repo'])
subprocess.run(['git', 'commit', '-m',
                'NB17: per-arch coupling bootstrap CIs + full EffNet dispersion'],
               cwd=CFG['repo'])
subprocess.run(['git', 'push'], cwd=CFG['repo'])
print('committed + pushed')

committed + pushed
